# Analyse des incohérences — déclarations de salaires CNPS

Ce notebook audite les fichiers Parquet bruts issus de l'étape 01 (`silver/cnps/`,
*avant* le nettoyage de l'étape 03 : aucun type de salarié exclu, aucun filtre de
salaire minimum, aucune winsorisation). C'est le point du pipeline le plus fidèle
aux déclarations telles que saisies, donc le plus pertinent pour repérer les
incohérences de saisie avant qu'elles ne soient corrigées ou filtrées en aval.

**Nécessite** `matplotlib` pour les graphiques (section 6) : `pip install matplotlib`
si besoin — le reste du notebook fonctionne sans.

**Contenu**
1. Chargement de tous les fichiers mensuels
2. Vue d'ensemble (volumétrie, valeurs manquantes)
3. `TYPE_SALARIE` : périodicité déclarée (Mensuel / Journalier / Horaire / non renseigné)
4. Salaires nuls, négatifs ou sous le seuil plausible selon la périodicité
5. Salaires probablement saisis dans la mauvaise unité (ex. un taux journalier déclaré comme mensuel)
6. Distribution du salaire (graphiques)
7. Salaire croisé avec le profil (secteur, taille d'entreprise, sexe, statut, commune)
8. Concentration des incohérences (entreprises et individus les plus touchés)
9. Autres incohérences utiles (durée travaillée, outliers hauts, variations mensuelles)
10. Synthèse récapitulative


In [ ]:
import sys
from pathlib import Path

import polars as pl

sys.path.insert(0, str(Path.cwd() / "src"))
from cnps.config import load_config
from cnps.storage import list_objects, read_parquet

cfg = load_config()
minio_cfg = cfg.minio

pl.Config.set_tbl_rows(30)


In [ ]:
# Graphiques (section 6) : necessite matplotlib (pip install matplotlib).
try:
    import matplotlib.pyplot as plt
    plt.rcParams["figure.figsize"] = (9, 5)
    plt.rcParams["axes.grid"] = True
    plt.rcParams["grid.alpha"] = 0.3
    _HAS_MPL = True
except ImportError:
    print("matplotlib non installe : les cellules de graphiques (section 6) seront ignorees.")
    _HAS_MPL = False


## 1. Chargement

Tous les fichiers `MM_AAAA.parquet` de `silver/cnps/` (sortie de l'étape 01), concaténés.

In [ ]:
objects = sorted(
    o for o in list_objects(minio_cfg, minio_cfg.processed_bucket, minio_cfg.processed_prefix, recursive=False)
    if o.endswith(".parquet")
)
print(f"{len(objects)} fichiers mensuels trouvés")

frames = []
for obj in objects:
    d = read_parquet(minio_cfg, minio_cfg.processed_bucket, obj)
    frames.append(d)

# Aligne les schemas (union des colonnes) avant concatenation, au cas ou
# un fichier mensuel aurait des colonnes en plus/en moins (cf. audit.py::_check_colonnes)
all_cols = dict.fromkeys(c for f in frames for c in f.columns)
aligned = []
for f in frames:
    for c in all_cols:
        if c not in f.columns:
            f = f.with_columns(pl.lit(None).alias(c))
    aligned.append(f.select(list(all_cols.keys())))

df = pl.concat(aligned, how="vertical_relaxed")

# SALAIRE_BRUT et DUREE_TRAVAILLEE sont lus en texte brut a l'etape 01
# (avant harmonisation de type de l'etape 02) : conversion numerique explicite ici.
df = df.with_columns(
    pl.col("SALAIRE_BRUT").cast(pl.Utf8).str.replace_all(r"[^\d.\-]", "").cast(pl.Float64, strict=False).alias("SALAIRE_BRUT"),
    pl.col("DUREE_TRAVAILLEE").cast(pl.Utf8).str.replace_all(r"[^\d.\-]", "").cast(pl.Float64, strict=False).alias("DUREE_TRAVAILLEE"),
)

print(f"{df.height:,} lignes, {df.width} colonnes")
df.select("ID_INDIV", "PERIOD", "TYPE_SALARIE", "SALAIRE_BRUT", "DUREE_TRAVAILLEE").head(5)


## 2. Vue d'ensemble

Volumétrie par mois et taux de valeurs manquantes sur les colonnes clés pour cette analyse.


In [ ]:
df.group_by("PERIOD").agg(
    pl.len().alias("n_declarations"),
    pl.col("ID_INDIV").n_unique().alias("n_individus"),
    pl.col("SALAIRE_BRUT").null_count().alias("n_salaire_manquant"),
).sort("PERIOD")


In [ ]:
cols_cles = ["SALAIRE_BRUT", "TYPE_SALARIE", "DUREE_TRAVAILLEE", "ID_INDIV", "ID_EMPLOYEUR"]
df.select(
    pl.len().alias("total_lignes"),
    *[pl.col(c).null_count().alias(f"na_{c}") for c in cols_cles],
)


## 3. `TYPE_SALARIE` : périodicité déclarée

`TYPE_SALARIE` code la périodicité de la déclaration :

| Code | Signification |
|------|----------------|
| `M`  | Salarié mensuel |
| `J`  | Salarié journalier |
| `H`  | Salarié horaire |
| *(vide)* | Non renseigné |

C'est cette colonne — pas une unité déduite du montant — qui indique en principe si
`SALAIRE_BRUT` correspond à un mois, un jour ou une heure de travail. Les incohérences
recherchées ici sont les cas où le **montant** ne correspond manifestement pas à la
**périodicité déclarée** (ou l'inverse, quand la périodicité est absente).


In [ ]:
type_salarie_counts = df["TYPE_SALARIE"].value_counts().sort("count", descending=True)
type_salarie_counts = type_salarie_counts.with_columns(
    (pl.col("count") / df.height * 100).round(2).alias("pct")
)
type_salarie_counts


In [ ]:
if _HAS_MPL:
    labels = {"M": "Mensuel", "J": "Journalier", "H": "Horaire", None: "Non renseigné"}
    plot_df = type_salarie_counts.to_pandas()
    plot_df["label"] = plot_df["TYPE_SALARIE"].map(labels).fillna(plot_df["TYPE_SALARIE"])

    fig, ax = plt.subplots()
    bars = ax.bar(plot_df["label"], plot_df["count"], color="#5DADE2", edgecolor="#2C3E50")
    ax.set_title("Répartition des déclarations par périodicité de salaire (TYPE_SALARIE)")
    ax.set_ylabel("Nombre de déclarations")
    for b, pct in zip(bars, plot_df["pct"]):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{pct:.1f}%",
                ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.show()


## 4. Salaires nuls, négatifs ou sous le seuil plausible

Le SMIG (salaire minimum) utilisé par le pipeline pour le nettoyage (étape 03) est de
**75 000 FCFA mensuel** (`cleaning.min_salary` dans `settings.yaml`). On en dérive des
seuils plausibles pour les périodicités journalière et horaire, sur la base de
**26 jours ouvrés/mois** et **8h/jour** (soit 208h/mois) — hypothèses standard,
modifiables ci-dessous si une autre convention est en vigueur à la CNPS.

Un salaire est considéré comme une **incohérence potentielle** s'il est :
- **nul ou manquant** alors que la ligne est par ailleurs déclarée (cas déjà couvert par
  la section valeurs manquantes, rappelé ici pour mémoire) ;
- **négatif** (impossible en toute circonstance) ;
- **inférieur au seuil plausible** de sa périodicité déclarée.


In [ ]:
SMIG_MENSUEL = cfg.cleaning.min_salary  # 75 000 FCFA, depuis settings.yaml
JOURS_OUVRES_PAR_MOIS = 26
HEURES_PAR_MOIS = 208  # 26 jours x 8h

SEUIL_PLAUSIBLE = {
    "M": float(SMIG_MENSUEL),
    "J": SMIG_MENSUEL / JOURS_OUVRES_PAR_MOIS,
    "H": SMIG_MENSUEL / HEURES_PAR_MOIS,
}
for code_, seuil in SEUIL_PLAUSIBLE.items():
    print(f"Seuil plausible pour TYPE_SALARIE='{code_}' : {seuil:,.0f} FCFA")


In [ ]:
df = df.with_columns(
    pl.col("TYPE_SALARIE").replace_strict(SEUIL_PLAUSIBLE, default=None, return_dtype=pl.Float64)
    .alias("SEUIL_PLAUSIBLE")
)

df = df.with_columns(
    pl.when(pl.col("SALAIRE_BRUT").is_null()).then(pl.lit("Manquant"))
    .when(pl.col("SALAIRE_BRUT") < 0).then(pl.lit("Négatif"))
    .when(pl.col("SALAIRE_BRUT") == 0).then(pl.lit("Nul (zéro)"))
    .when(
        pl.col("SEUIL_PLAUSIBLE").is_not_null() & (pl.col("SALAIRE_BRUT") < pl.col("SEUIL_PLAUSIBLE"))
    ).then(pl.lit("Sous le seuil plausible"))
    .otherwise(pl.lit("OK"))
    .alias("STATUT_SALAIRE")
)

statut_counts = df["STATUT_SALAIRE"].value_counts().sort("count", descending=True)
statut_counts = statut_counts.with_columns((pl.col("count") / df.height * 100).round(2).alias("pct"))
statut_counts


In [ ]:
print("Répartition des anomalies de salaire par périodicité déclarée :")
(
    df.filter(pl.col("STATUT_SALAIRE") != "OK")
    .group_by("TYPE_SALARIE", "STATUT_SALAIRE")
    .agg(pl.len().alias("n"))
    .sort(["TYPE_SALARIE", "n"], descending=[False, True])
)


In [ ]:
if _HAS_MPL:
    fig, ax = plt.subplots()
    anomalies = df.filter(pl.col("STATUT_SALAIRE") != "OK")["STATUT_SALAIRE"].value_counts().sort("count", descending=True).to_pandas()
    colors = {"Manquant": "#AEB6BF", "Négatif": "#C0392B", "Nul (zéro)": "#E67E22", "Sous le seuil plausible": "#F1C40F"}
    bar_colors = [colors.get(s, "#5DADE2") for s in anomalies["STATUT_SALAIRE"]]
    ax.barh(anomalies["STATUT_SALAIRE"], anomalies["count"], color=bar_colors, edgecolor="#2C3E50")
    ax.set_title("Anomalies de SALAIRE_BRUT (hors 'OK')")
    ax.set_xlabel("Nombre de déclarations")
    plt.tight_layout()
    plt.show()


## 5. Salaires probablement saisis dans la mauvaise unité

Au-delà du seuil minimal, un signal plus spécifique d'erreur de périodicité est un
salaire **mensuel** (`TYPE_SALARIE == 'M'`) dont le montant est du même ordre de
grandeur qu'un taux **journalier** ou **horaire** plausible — ou inversement, un salaire
**journalier/horaire** dont le montant est du même ordre qu'un salaire mensuel complet.
C'est le signal le plus direct d'une confusion d'unité à la saisie.


In [ ]:
# Salaire "M" mais montant compatible avec un taux journalier ou horaire plausible
# (entre le seuil horaire et un maximum journalier raisonnable, ex. 3x le seuil journalier)
mensuel_suspect = df.filter(
    (pl.col("TYPE_SALARIE") == "M")
    & pl.col("SALAIRE_BRUT").is_not_null()
    & (pl.col("SALAIRE_BRUT") > 0)
    & (pl.col("SALAIRE_BRUT") < SEUIL_PLAUSIBLE["J"] * 3)
)
print(f"Déclarations 'Mensuel' avec un montant proche d'un taux journalier/horaire : {mensuel_suspect.height:,}")

# Salaire "J" ou "H" mais montant compatible avec un salaire mensuel complet
journalier_horaire_suspect = df.filter(
    pl.col("TYPE_SALARIE").is_in(["J", "H"])
    & pl.col("SALAIRE_BRUT").is_not_null()
    & (pl.col("SALAIRE_BRUT") >= SMIG_MENSUEL)
)
print(f"Déclarations 'Journalier'/'Horaire' avec un montant proche d'un salaire mensuel complet : {journalier_horaire_suspect.height:,}")


In [ ]:
mensuel_suspect.select(
    "PERIOD", "ID_INDIV", "ID_EMPLOYEUR", "TYPE_SALARIE", "SALAIRE_BRUT", "DUREE_TRAVAILLEE"
).sort("SALAIRE_BRUT", descending=True).head(15)


In [ ]:
journalier_horaire_suspect.select(
    "PERIOD", "ID_INDIV", "ID_EMPLOYEUR", "TYPE_SALARIE", "SALAIRE_BRUT", "DUREE_TRAVAILLEE"
).sort("SALAIRE_BRUT", descending=True).head(15)


## 6. Distribution du salaire

Lu sur `SALAIRE_BRUT` brut (avant nettoyage/winsorisation), par périodicité déclarée. Nécessite `matplotlib`.

In [ ]:
if _HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    type_labels = {"M": "Mensuel", "J": "Journalier", "H": "Horaire"}
    colors_hist = {"M": "#5DADE2", "J": "#58D68D", "H": "#AF7AC5"}

    for ax, (code_, label) in zip(axes, type_labels.items()):
        serie = df.filter(
            (pl.col("TYPE_SALARIE") == code_)
            & pl.col("SALAIRE_BRUT").is_not_null()
            & (pl.col("SALAIRE_BRUT") > 0)
        )["SALAIRE_BRUT"]
        if serie.len() == 0:
            continue
        # Ecrete a p99 pour la lisibilite du graphique (n'affecte pas les donnees)
        p99 = serie.quantile(0.99)
        ax.hist(serie.filter(serie <= p99).to_numpy(), bins=60, color=colors_hist[code_], edgecolor="white")
        ax.axvline(SEUIL_PLAUSIBLE[code_], color="#C0392B", linestyle="--", linewidth=1.5,
                   label=f"Seuil plausible ({SEUIL_PLAUSIBLE[code_]:,.0f})")
        ax.set_title(f"{label} (n={serie.len():,}, écrêté à p99)")
        ax.set_xlabel("SALAIRE_BRUT (FCFA)")
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()


In [ ]:
if _HAS_MPL:
    fig, ax = plt.subplots(figsize=(9, 5))
    data_box = [
        df.filter((pl.col("TYPE_SALARIE") == c) & pl.col("SALAIRE_BRUT").is_not_null() & (pl.col("SALAIRE_BRUT") > 0))["SALAIRE_BRUT"].to_numpy()
        for c in ["M", "J", "H"]
    ]
    bp = ax.boxplot(data_box, tick_labels=["Mensuel", "Journalier", "Horaire"], showfliers=False, patch_artist=True)
    for patch, color in zip(bp["boxes"], ["#5DADE2", "#58D68D", "#AF7AC5"]):
        patch.set_facecolor(color)
    ax.set_yscale("log")
    ax.set_title("Distribution de SALAIRE_BRUT par périodicité (échelle log, sans valeurs extrêmes)")
    ax.set_ylabel("SALAIRE_BRUT (FCFA, échelle log)")
    plt.tight_layout()
    plt.show()


In [ ]:
if _HAS_MPL:
    fig, ax = plt.subplots()
    evol = (
        df.filter(pl.col("SALAIRE_BRUT").is_not_null() & (pl.col("SALAIRE_BRUT") > 0))
        .group_by("PERIOD")
        .agg(
            pl.col("SALAIRE_BRUT").median().alias("mediane"),
            pl.col("SALAIRE_BRUT").quantile(0.25).alias("q1"),
            pl.col("SALAIRE_BRUT").quantile(0.75).alias("q3"),
        )
        .sort("PERIOD")
        .to_pandas()
    )
    ax.plot(evol["PERIOD"], evol["mediane"], color="#2C3E50", marker="o", label="Médiane")
    ax.fill_between(evol["PERIOD"], evol["q1"], evol["q3"], color="#5DADE2", alpha=0.3, label="Q1–Q3")
    ax.set_title("Évolution mensuelle de SALAIRE_BRUT (toutes périodicités confondues)")
    ax.set_ylabel("FCFA")
    ax.tick_params(axis="x", rotation=90)
    ax.legend()
    plt.tight_layout()
    plt.show()


## 7. Salaire croisé avec le profil

Les sections précédentes traitent les incohérences de salaire de façon globale. Ici, on
croise `SALAIRE_BRUT` (et les anomalies de la section 4) avec les variables de profil
disponibles dans les fichiers bruts — pour voir si les incohérences sont **diffuses**
(réparties partout, plutôt un problème de saisie général) ou **concentrées** dans
certains segments (secteur, taille d'entreprise, statut...), ce qui orienterait vers
une cause spécifique (ex. un formulaire mal adapté à un type d'employeur).


### 7.1 Par secteur d'activité

In [ ]:
par_secteur = (
    df.group_by("SECTEUR_ACTIVITE")
    .agg(
        pl.len().alias("n"),
        pl.col("SALAIRE_BRUT").is_not_null().sum().alias("n_salaire_renseigne"),
        pl.col("SALAIRE_BRUT").median().alias("salaire_median"),
        (pl.col("STATUT_SALAIRE") != "OK").sum().alias("n_anomalies"),
    )
    .with_columns((pl.col("n_anomalies") / pl.col("n") * 100).round(2).alias("pct_anomalies"))
    .sort("n", descending=True)
)
par_secteur.head(20)


In [ ]:
if _HAS_MPL:
    top = par_secteur.filter(pl.col("n") >= 500).sort("pct_anomalies", descending=True).head(15).to_pandas()
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top["SECTEUR_ACTIVITE"], top["pct_anomalies"], color="#E67E22", edgecolor="#2C3E50")
    ax.set_title("Taux d'anomalies de salaire par secteur (secteurs avec ≥500 déclarations)")
    ax.set_xlabel("% de lignes en anomalie")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


### 7.2 Par taille d'entreprise (`EFFECTIF_SALARIES`)

In [ ]:
_TAILLE_BREAKS = [
    (0, 2, "1 salarié"), (2, 11, "2-10"), (11, 51, "11-50"),
    (51, 201, "51-200"), (201, 1001, "201-1000"), (1001, 10**9, "1000+"),
]
taille_expr = pl.when(pl.col("EFFECTIF_SALARIES").is_null()).then(pl.lit(None).cast(pl.Utf8))
for lo, hi, label in _TAILLE_BREAKS:
    taille_expr = taille_expr.when(
        (pl.col("EFFECTIF_SALARIES") >= lo) & (pl.col("EFFECTIF_SALARIES") < hi)
    ).then(pl.lit(label))
taille_expr = taille_expr.otherwise(pl.lit(None).cast(pl.Utf8))

df = df.with_columns(taille_expr.alias("CLASSE_EFFECTIF_NB"))

par_taille = (
    df.group_by("CLASSE_EFFECTIF_NB")
    .agg(
        pl.len().alias("n"),
        pl.col("SALAIRE_BRUT").median().alias("salaire_median"),
        (pl.col("STATUT_SALAIRE") != "OK").sum().alias("n_anomalies"),
    )
    .with_columns((pl.col("n_anomalies") / pl.col("n") * 100).round(2).alias("pct_anomalies"))
    .sort("n", descending=True)
)
par_taille


### 7.3 Par sexe

In [ ]:
par_sexe = (
    df.group_by("SEXE")
    .agg(
        pl.len().alias("n"),
        pl.col("SALAIRE_BRUT").median().alias("salaire_median"),
        pl.col("SALAIRE_BRUT").mean().round(0).alias("salaire_moyen"),
        (pl.col("STATUT_SALAIRE") != "OK").sum().alias("n_anomalies"),
    )
    .with_columns((pl.col("n_anomalies") / pl.col("n") * 100).round(2).alias("pct_anomalies"))
)
par_sexe


### 7.4 Par statut du travailleur (`STATUT_TRAVAILLEUR`)

In [ ]:
par_statut = (
    df.group_by("STATUT_TRAVAILLEUR")
    .agg(
        pl.len().alias("n"),
        pl.col("SALAIRE_BRUT").median().alias("salaire_median"),
        (pl.col("STATUT_SALAIRE") != "OK").sum().alias("n_anomalies"),
    )
    .with_columns((pl.col("n_anomalies") / pl.col("n") * 100).round(2).alias("pct_anomalies"))
    .sort("n", descending=True)
)
par_statut


### 7.5 Par commune (top 15 par volume)

In [ ]:
par_commune = (
    df.group_by("COMMUNE")
    .agg(
        pl.len().alias("n"),
        pl.col("SALAIRE_BRUT").median().alias("salaire_median"),
        (pl.col("STATUT_SALAIRE") != "OK").sum().alias("n_anomalies"),
    )
    .with_columns((pl.col("n_anomalies") / pl.col("n") * 100).round(2).alias("pct_anomalies"))
    .sort("n", descending=True)
)
par_commune.head(15)


In [ ]:
if _HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    top_taille = par_taille.filter(pl.col("CLASSE_EFFECTIF_NB").is_not_null()).to_pandas()
    axes[0].bar(top_taille["CLASSE_EFFECTIF_NB"], top_taille["salaire_median"], color="#5DADE2", edgecolor="#2C3E50")
    axes[0].set_title("Salaire médian par taille d'entreprise")
    axes[0].set_ylabel("SALAIRE_BRUT médian (FCFA)")
    axes[0].tick_params(axis="x", rotation=30)

    top_sect = par_secteur.filter(pl.col("n") >= 500).sort("salaire_median", descending=True).head(10).to_pandas()
    axes[1].barh(top_sect["SECTEUR_ACTIVITE"], top_sect["salaire_median"], color="#58D68D", edgecolor="#2C3E50")
    axes[1].set_title("Salaire médian par secteur (top 10, ≥500 décl.)")
    axes[1].set_xlabel("SALAIRE_BRUT médian (FCFA)")
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.show()


## 8. Concentration des incohérences

Les anomalies de salaire sont-elles diluées sur l'ensemble des employeurs et des
individus, ou concentrées sur un petit nombre d'entre eux ? Une forte concentration
oriente vers une action ciblée (relance de quelques entreprises) plutôt qu'une
correction généralisée.


### 8.1 Entreprises avec le plus de déclarations en anomalie

In [ ]:
par_employeur = (
    df.filter(pl.col("ID_EMPLOYEUR").is_not_null())
    .group_by("ID_EMPLOYEUR", "RAISON_SOCIALE")
    .agg(
        pl.len().alias("n_declarations"),
        (pl.col("STATUT_SALAIRE") != "OK").sum().alias("n_anomalies"),
    )
    .filter(pl.col("n_anomalies") > 0)
    .with_columns((pl.col("n_anomalies") / pl.col("n_declarations") * 100).round(2).alias("pct_anomalies"))
    .sort("n_anomalies", descending=True)
)
print(f"{par_employeur.height:,} entreprises ont au moins une déclaration en anomalie de salaire.")
par_employeur.head(15)


In [ ]:
# Concentration : part des anomalies totales portee par les employeurs les plus touches
n_anomalies_total = par_employeur["n_anomalies"].sum()
for pct_top in [1, 5, 10]:
    n_top = max(1, int(par_employeur.height * pct_top / 100))
    part = par_employeur.sort("n_anomalies", descending=True).head(n_top)["n_anomalies"].sum()
    print(f"Top {pct_top}% des employeurs concernés ({n_top} entreprises) "
          f"portent {part / n_anomalies_total * 100:.1f}% des anomalies de salaire.")


### 8.2 Individus avec des anomalies récurrentes (plusieurs mois)

In [ ]:
par_individu = (
    df.filter(pl.col("ID_INDIV").is_not_null())
    .group_by("ID_INDIV")
    .agg(
        pl.len().alias("n_declarations"),
        (pl.col("STATUT_SALAIRE") != "OK").sum().alias("n_anomalies"),
    )
    .filter(pl.col("n_anomalies") >= 2)
    .with_columns((pl.col("n_anomalies") / pl.col("n_declarations") * 100).round(2).alias("pct_anomalies"))
    .sort("n_anomalies", descending=True)
)
print(f"{par_individu.height:,} individus ont au moins 2 déclarations en anomalie de salaire "
      f"(pas un incident isolé, mais un motif récurrent).")
par_individu.head(15)


In [ ]:
if _HAS_MPL:
    fig, ax = plt.subplots()
    dist_n_anomalies = par_employeur["n_anomalies"].value_counts().sort("n_anomalies")
    ax.bar(dist_n_anomalies["n_anomalies"].to_numpy(), dist_n_anomalies["count"].to_numpy(),
           color="#C0392B", edgecolor="#2C3E50")
    ax.set_title("Nombre d'entreprises par volume d'anomalies de salaire portées")
    ax.set_xlabel("Nombre d'anomalies portées par l'entreprise")
    ax.set_ylabel("Nombre d'entreprises")
    ax.set_yscale("log")
    plt.tight_layout()
    plt.show()


## 9. Autres incohérences utiles

Quelques contrôles complémentaires à ceux d'`audit.py`, centrés spécifiquement sur le
salaire et sa cohérence avec les autres variables de la déclaration.


### 9.1 `DUREE_TRAVAILLEE` incohérente avec `TYPE_SALARIE`

Une durée > 31 n'a de sens pour aucune périodicité ; une durée à 0 avec un salaire positif est également suspecte.

In [ ]:
duree_suspecte = df.filter(
    (pl.col("DUREE_TRAVAILLEE") > 31)
    | ((pl.col("DUREE_TRAVAILLEE") == 0) & pl.col("SALAIRE_BRUT").is_not_null() & (pl.col("SALAIRE_BRUT") > 0))
)
print(f"Lignes avec DUREE_TRAVAILLEE incohérente : {duree_suspecte.height:,}")
duree_suspecte.select("PERIOD", "ID_INDIV", "TYPE_SALARIE", "SALAIRE_BRUT", "DUREE_TRAVAILLEE").head(10)


### 9.2 Salaires extrêmement élevés (outliers hauts)

Au-delà du seuil bas, les montants excessifs (> 50 000 000 FCFA, borne haute de `estimation.salary_plausible_range`) méritent aussi vérification.

In [ ]:
SALAIRE_MAX_PLAUSIBLE = cfg.estimation.salary_plausible_range[1]
salaires_excessifs = df.filter(pl.col("SALAIRE_BRUT") > SALAIRE_MAX_PLAUSIBLE)
print(f"Seuil haut plausible : {SALAIRE_MAX_PLAUSIBLE:,.0f} FCFA")
print(f"Lignes au-delà du seuil : {salaires_excessifs.height:,}")
salaires_excessifs.select("PERIOD", "ID_INDIV", "ID_EMPLOYEUR", "TYPE_SALARIE", "SALAIRE_BRUT").sort("SALAIRE_BRUT", descending=True).head(10)


### 9.3 Individus avec un salaire très variable d'un mois à l'autre

Pour un même `ID_INDIV`, un salaire qui double ou est divisé par deux d'un mois sur l'autre peut signaler une erreur de saisie ponctuelle (plutôt qu'une réelle évolution de carrière).

In [ ]:
var_indiv = (
    df.filter(pl.col("SALAIRE_BRUT").is_not_null() & (pl.col("SALAIRE_BRUT") > 0))
    .sort(["ID_INDIV", "PERIOD"])
    .with_columns(
        pl.col("SALAIRE_BRUT").shift(1).over("ID_INDIV").alias("SALAIRE_PRECEDENT"),
        pl.col("PERIOD").shift(1).over("ID_INDIV").alias("PERIOD_PRECEDENT"),
    )
    .filter(pl.col("SALAIRE_PRECEDENT").is_not_null())
    .with_columns(
        (pl.col("SALAIRE_BRUT") / pl.col("SALAIRE_PRECEDENT")).alias("RATIO")
    )
)
variations_fortes = var_indiv.filter((pl.col("RATIO") >= 3) | (pl.col("RATIO") <= 1 / 3))
print(f"Transitions mois-à-mois avec un salaire multiplié/divisé par 3 ou plus : {variations_fortes.height:,} "
      f"({variations_fortes.height / var_indiv.height * 100:.2f}% des transitions observées)")
variations_fortes.select(
    "ID_INDIV", "PERIOD_PRECEDENT", "SALAIRE_PRECEDENT", "PERIOD", "SALAIRE_BRUT", "RATIO"
).sort("RATIO", descending=True).head(10)


## 10. Synthèse récapitulative

Tableau de bord final regroupant les volumes de chaque incohérence identifiée ci-dessus.


In [ ]:
synthese = pl.DataFrame([
    {"Incohérence": "Salaire manquant", "n": df.filter(pl.col("STATUT_SALAIRE") == "Manquant").height},
    {"Incohérence": "Salaire négatif", "n": df.filter(pl.col("STATUT_SALAIRE") == "Négatif").height},
    {"Incohérence": "Salaire nul (zéro)", "n": df.filter(pl.col("STATUT_SALAIRE") == "Nul (zéro)").height},
    {"Incohérence": "Salaire sous le seuil plausible (selon périodicité)", "n": df.filter(pl.col("STATUT_SALAIRE") == "Sous le seuil plausible").height},
    {"Incohérence": "TYPE_SALARIE non renseigné", "n": df.filter(pl.col("TYPE_SALARIE").is_null()).height},
    {"Incohérence": "Mensuel avec montant proche d'un taux journalier/horaire", "n": mensuel_suspect.height},
    {"Incohérence": "Journalier/Horaire avec montant proche d'un salaire mensuel", "n": journalier_horaire_suspect.height},
    {"Incohérence": "Entreprises avec au moins une anomalie de salaire", "n": par_employeur.height},
    {"Incohérence": "Individus avec anomalies récurrentes (≥2 mois)", "n": par_individu.height},
    {"Incohérence": "DUREE_TRAVAILLEE incohérente", "n": duree_suspecte.height},
    {"Incohérence": "Salaire au-delà du seuil haut plausible", "n": salaires_excessifs.height},
    {"Incohérence": "Variation mois-à-mois x3 ou plus (même individu)", "n": variations_fortes.height},
]).with_columns((pl.col("n") / df.height * 100).round(2).alias("pct_du_total"))

synthese


### Constats de concentration

Rappel des deux indicateurs de concentration calculés en section 8, utiles pour prioriser
une action corrective (cibler un petit nombre d'entreprises plutôt qu'une correction
généralisée).


In [ ]:
print(f"- {par_employeur.height:,} entreprises portent au moins une anomalie de salaire.")
n_top5 = max(1, int(par_employeur.height * 0.05))
part_top5 = par_employeur.sort("n_anomalies", descending=True).head(n_top5)["n_anomalies"].sum()
print(f"- Le top 5% de ces entreprises ({n_top5}) concentre {part_top5 / n_anomalies_total * 100:.1f}% des anomalies.")
print(f"- {par_individu.height:,} individus ont des anomalies récurrentes (≥2 déclarations sur des mois différents).")


In [ ]:
if _HAS_MPL:
    fig, ax = plt.subplots(figsize=(9, 6))
    s = synthese.sort("n").to_pandas()
    ax.barh(s["Incohérence"], s["n"], color="#5DADE2", edgecolor="#2C3E50")
    ax.set_title("Synthèse des incohérences relevées")
    ax.set_xlabel("Nombre de lignes concernées")
    plt.tight_layout()
    plt.show()
